# DMEPOS by Referring Provider data loading and cleaning

This notebook combines and cleans the Medicare Durable Medical Equipment, Devices & Supplies (DMEPOS) - by Referring Provider datasets from the Centers of Medicare & Medicaid Services (CMS) for program years 2021 through 2023. Data cleaning and preprocessing steps include the imputation of missing values, normalization of text fields, and appending program year to each record. Finally, data field headings are transformed into snake case for consistency with other datasets.

For information regarding each variable, please refer to the data dictionary downloadable from [DMEPOS - by Referring Provider](https://data.cms.gov/provider-summary-by-type-of-service/medicare-durable-medical-equipment-devices-supplies/medicare-durable-medical-equipment-devices-supplies-by-referring-provider).

Cells as NB Raw Convert are not necessary to run to understand work performed nor run this notebook without exception

## Import raw files

In [1]:
import pandas as pd

files_names = ['/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v20_dy21_rfrr.csv',
               '/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v20_dy22_rfrr.csv',
               '/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v10_dy23_rfrr.csv']

rfrr = {}

dy = 20
for file_name in files_names:
    dy += 1

    # Create DataFrame from the file content
    df = pd.read_csv(file_name,dtype={'Rfrg_Prvdr_State_FIPS':str,'Rfrg_Prvdr_Zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str
    rfrr[dy] = df

Verify the number of records in each file.

In [2]:
len21 = len(rfrr[21])
len22 = len(rfrr[22])
len23 = len(rfrr[23])
print(f'{len21}, {len22}, {len23}')

384264, 385637, 387352


### Change these to code cells and run for additional validation.

## Combine and Clean Datasets

In [3]:
# combine dfs
df_raw = pd.concat([rfrr[21],rfrr[22],rfrr[23]],axis=0)
df_raw.head()

,Rfrg_NPI,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,Rfrg_Prvdr_State_Abrvtn,...,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre
0,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,NaN,NaN,NaN,1.000000,NaN,NaN,0.0,NaN,NaN,1.942167
1,1003000480,Rothchild,Kevin,B,MD,I,12605 E 16th Ave,NaN,Aurora,CO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.278200
2,1003000522,Weigand,Frederick,J,MD,I,1565 Saxon Blvd,Suite 102,Deltona,FL,...,NaN,NaN,1.000000,0.846154,NaN,NaN,0.0,NaN,NaN,1.872308
3,1003000530,Semonche,Amanda,M,DO,I,1021 Park Ave,Suite 203,Quakertown,PA,...,0.722222,NaN,0.833333,0.777778,NaN,NaN,NaN,NaN,NaN,2.144684
4,1003000597,Kim,Dae,Y,"M.D., PH.D",I,1145 S Utica Ave,Suite 202,Tulsa,OK,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.006500


### Convert the following cell to code an run to see data types.

In [4]:
# check columns which have missing data
pd.set_option('display.max_rows', None)
df_nulltots = df_raw.isnull().sum()
df_nulltots

Rfrg_NPI                                  0
Rfrg_Prvdr_Last_Name_Org                  0
Rfrg_Prvdr_First_Name                    16
Rfrg_Prvdr_MI                        359299
Rfrg_Prvdr_Crdntls                    66734
Rfrg_Prvdr_Ent_Cd                         0
Rfrg_Prvdr_St1                            0
Rfrg_Prvdr_St2                       866550
Rfrg_Prvdr_City                           0
Rfrg_Prvdr_State_Abrvtn                   0
Rfrg_Prvdr_State_FIPS                     0
Rfrg_Prvdr_Zip5                           0
Rfrg_Prvdr_RUCA                        1080
Rfrg_Prvdr_RUCA_Desc                   1080
Rfrg_Prvdr_Cntry                          0
Rfrg_Prvdr_Spclty_Desc                    0
Rfrg_Prvdr_Spclty_Srce                    0
Tot_Suplrs                                0
Tot_Suplr_HCPCS_Cds                       0
Tot_Suplr_Benes                      496685
Tot_Suplr_Clms                            0
Tot_Suplr_Srvcs                           0
Suplr_Sbmtd_Chrgs               

### Explore null values to determine best method of imputation.

As many beneficiary demographic and diagnosis fields are null for a majority of records, we determine that there is not a rational way to impute null values. From subsequent analysis, we determined that knowledge that a field was reported null for a given provider may be informative as to whether the provider would be subsequently excluded for fraud. The method of imputation for null DME, POS, and Drug fields is deemed arbitrary as these are aggregates of more granular information that we obtain from the referring provider and service dataset. Hence, we convert the subsequent cell to Raw NBConvert and maintain for documentation purposes only.

### Investigation of null specialty codes

In [5]:
# explore missing credentials
# lack of credential may be appropriate for 'Student in an Organized Health Care Education/Training Program' or 'Hospital' but appears inappropriate for 'General Surgery'
# as missing credentials could be indicative of fraud, we will not impute missing values.
df_nocrd = df_raw[df_raw.Rfrg_Prvdr_Crdntls.isnull()]
df_nocrd.Rfrg_Prvdr_Spclty_Desc.unique()

array(['Physician Assistant', 'General Surgery', 'Neurology',
       'Nephrology', 'Nurse Practitioner', 'Hematology-Oncology',
       'Family Practice', 'Hospitalist', 'Cardiology', 'Endocrinology',
       'Emergency Medicine', 'Ophthalmology', 'Hand Surgery',
       'Internal Medicine', 'Sports Medicine', 'Orthopedic Surgery',
       'Physical Medicine and Rehabilitation', 'Medical Oncology',
       'Podiatry',
       'Student in an Organized Health Care Education/Training Program',
       'Pulmonary Disease', 'Otolaryngology', 'Hematology',
       'Pediatric Medicine', 'Critical Care (Intensivists)',
       'Osteopathic Manipulative Medicine', 'Rheumatology',
       'Certified Clinical Nurse Specialist', 'Neurosurgery',
       'General Practice', 'Urology', 'Obstetrics & Gynecology',
       'Vascular Surgery', 'Optometry', 'Geriatric Medicine',
       'Colorectal Surgery (Proctology)', 'Diagnostic Radiology',
       'Infectious Disease', 'Family Medicine', 'Gastroenterology',
      

### Exploration of supressed claims

This exploration was conducted to determine a method of imputing null DME fields, but as discussed previously, this is arbitrary as these fields merely aggregate more granular information that we obtain from the referring provider and service dataset. We convert cells to Raw NBConvert and maintain for documentation purposes only.

About 4/5 of total beneficiaries in the dataset are for DME claims, about 1/5 are for POS claims, and the remainder are for drug claims. As such, I elect to impute missing DME_Tot_Suplr_Benes data as 4 and POS_Tot_Suplr_Benes as 1 keeping in-line with imputing 5 for Tot_Suplr_Benes as done in the referring provider and servcie dataset. Missing Drug claims data will be imputed as zero, as total beneficiaries for drug claims account only 2% of the total beneficiaries in the dataset. After computing median values for POS claims with null Tot_Suplr_Benes, and finding all were zero, I elected to impute missing DME claim beneficiaries as 5 and impute other values with their median for records with null Tot_Suplr_Benes. All missing POS and Drug claims data will be imputed as zero.

In [6]:
# calculate medians of non-null Durable Medical Equipment data where Tot_Suplr_Benes is null
DME_Tot_Suplrs_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['DME_Tot_Suplrs'].median()
DME_Tot_Suplr_HCPCS_Cds_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['DME_Tot_Suplr_HCPCS_Cds'].median()
DME_Tot_Suplr_Clms_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['DME_Tot_Suplr_Clms'].median()
DME_Tot_Suplr_Srvcs_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['DME_Tot_Suplr_Srvcs'].median()
DME_Suplr_Sbmtd_Chrgs_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['DME_Suplr_Sbmtd_Chrgs'].median()
DME_Suplr_Mdcr_Alowd_Amt_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['DME_Suplr_Mdcr_Alowd_Amt'].median()
DME_Suplr_Mdcr_Pymt_Amt_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['DME_Suplr_Mdcr_Pymt_Amt'].median()
DME_Suplr_Mdcr_Stdzd_Pymt_Amt_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['DME_Suplr_Mdcr_Stdzd_Pymt_Amt'].median()

print(f'{DME_Tot_Suplrs_med}, {DME_Tot_Suplr_HCPCS_Cds_med}, {DME_Tot_Suplr_Clms_med}, {DME_Tot_Suplr_Srvcs_med}, {DME_Suplr_Sbmtd_Chrgs_med}, {DME_Suplr_Mdcr_Alowd_Amt_med}, {DME_Suplr_Mdcr_Pymt_Amt_med}, {DME_Suplr_Mdcr_Stdzd_Pymt_Amt_med}')

# calculate medians of non-null Prosthetic and Orthotic Suppression data where Tot_Suplr_Benes is null
POS_Tot_Suplrs_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['POS_Tot_Suplrs'].median()
POS_Tot_Suplr_HCPCS_Cds_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['POS_Tot_Suplr_HCPCS_Cds'].median()
POS_Tot_Suplr_Clms_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['POS_Tot_Suplr_Clms'].median()
POS_Tot_Suplr_Srvcs_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['POS_Tot_Suplr_Srvcs'].median()
POS_Suplr_Sbmtd_Chrgs_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['POS_Suplr_Sbmtd_Chrgs'].median()
POS_Suplr_Mdcr_Alowd_Amt_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['POS_Suplr_Mdcr_Alowd_Amt'].median()
POS_Suplr_Mdcr_Pymt_Amt_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['POS_Suplr_Mdcr_Pymt_Amt'].median()
POS_Suplr_Mdcr_Stdzd_Pymt_Amt_med = df_raw[df_raw.Tot_Suplr_Benes.isnull()]['POS_Suplr_Mdcr_Stdzd_Pymt_Amt'].median()

print(f'{POS_Tot_Suplrs_med}, {POS_Tot_Suplr_HCPCS_Cds_med}, {POS_Tot_Suplr_Clms_med}, {POS_Tot_Suplr_Srvcs_med}, {POS_Suplr_Sbmtd_Chrgs_med}, {POS_Suplr_Mdcr_Alowd_Amt_med}, {POS_Suplr_Mdcr_Pymt_Amt_med}, {POS_Suplr_Mdcr_Stdzd_Pymt_Amt_med}')



3.0, 6.0, 19.0, 43.0, 6075.045, 1947.45, 1449.27, 1540.0149999999999
0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0


In [7]:
# impute missing DME values
impute_vals = {'DME_Tot_Suplrs':DME_Tot_Suplrs_med, 'DME_Tot_Suplr_HCPCS_Cds':DME_Tot_Suplr_HCPCS_Cds_med,'DME_Tot_Suplr_Benes':5.,
               'DME_Tot_Suplr_Clms':DME_Tot_Suplr_Clms_med,'DME_Tot_Suplr_Srvcs':DME_Tot_Suplr_Srvcs_med,
               'DME_Suplr_Sbmtd_Chrgs':DME_Suplr_Sbmtd_Chrgs_med,'DME_Suplr_Mdcr_Alowd_Amt':DME_Suplr_Mdcr_Alowd_Amt_med,
               'DME_Suplr_Mdcr_Pymt_Amt':DME_Suplr_Mdcr_Pymt_Amt_med,'DME_Suplr_Mdcr_Stdzd_Pymt_Amt':DME_Suplr_Mdcr_Stdzd_Pymt_Amt_med}

df_clean = df_raw.fillna(impute_vals)
df_clean.isnull().sum()

Rfrg_NPI                                  0
Rfrg_Prvdr_Last_Name_Org                  0
Rfrg_Prvdr_First_Name                    16
Rfrg_Prvdr_MI                        359299
Rfrg_Prvdr_Crdntls                    66734
Rfrg_Prvdr_Ent_Cd                         0
Rfrg_Prvdr_St1                            0
Rfrg_Prvdr_St2                       866550
Rfrg_Prvdr_City                           0
Rfrg_Prvdr_State_Abrvtn                   0
Rfrg_Prvdr_State_FIPS                     0
Rfrg_Prvdr_Zip5                           0
Rfrg_Prvdr_RUCA                        1080
Rfrg_Prvdr_RUCA_Desc                   1080
Rfrg_Prvdr_Cntry                          0
Rfrg_Prvdr_Spclty_Desc                    0
Rfrg_Prvdr_Spclty_Srce                    0
Tot_Suplrs                                0
Tot_Suplr_HCPCS_Cds                       0
Tot_Suplr_Benes                      496685
Tot_Suplr_Clms                            0
Tot_Suplr_Srvcs                           0
Suplr_Sbmtd_Chrgs               

In [8]:
# impute 0 to missing POS and Drug data (but skip sprsn indicators)
# impute -1 to missing demographic and diagnosis data
df_clean.loc[:,'POS_Tot_Suplrs':'POS_Suplr_Mdcr_Stdzd_Pymt_Amt'] = df_raw.loc[:,'POS_Tot_Suplrs':'POS_Suplr_Mdcr_Stdzd_Pymt_Amt'].fillna(0.)
df_clean.loc[:,'Drug_Tot_Suplrs':'Drug_Suplr_Mdcr_Stdzd_Pymt_Amt'] = df_raw.loc[:,'Drug_Tot_Suplrs':'Drug_Suplr_Mdcr_Stdzd_Pymt_Amt'].fillna(0.)
df_clean.loc[:,'Bene_Age_LT_65_Cnt':'Bene_Avg_Risk_Scre'] = df_raw.loc[:,'Bene_Age_LT_65_Cnt':'Bene_Avg_Risk_Scre'].fillna(-1.)
df_clean.isnull().sum()

Rfrg_NPI                                  0
Rfrg_Prvdr_Last_Name_Org                  0
Rfrg_Prvdr_First_Name                    16
Rfrg_Prvdr_MI                        359299
Rfrg_Prvdr_Crdntls                    66734
Rfrg_Prvdr_Ent_Cd                         0
Rfrg_Prvdr_St1                            0
Rfrg_Prvdr_St2                       866550
Rfrg_Prvdr_City                           0
Rfrg_Prvdr_State_Abrvtn                   0
Rfrg_Prvdr_State_FIPS                     0
Rfrg_Prvdr_Zip5                           0
Rfrg_Prvdr_RUCA                        1080
Rfrg_Prvdr_RUCA_Desc                   1080
Rfrg_Prvdr_Cntry                          0
Rfrg_Prvdr_Spclty_Desc                    0
Rfrg_Prvdr_Spclty_Srce                    0
Tot_Suplrs                                0
Tot_Suplr_HCPCS_Cds                       0
Tot_Suplr_Benes                      496685
Tot_Suplr_Clms                            0
Tot_Suplr_Srvcs                           0
Suplr_Sbmtd_Chrgs               

In [9]:
# impute missing Tot_Suplr_Benes as 5
df_clean['Tot_Suplr_Benes'].fillna(5, inplace=True)
df_clean.isnull().sum()

Rfrg_NPI                                  0
Rfrg_Prvdr_Last_Name_Org                  0
Rfrg_Prvdr_First_Name                    16
Rfrg_Prvdr_MI                        359299
Rfrg_Prvdr_Crdntls                    66734
Rfrg_Prvdr_Ent_Cd                         0
Rfrg_Prvdr_St1                            0
Rfrg_Prvdr_St2                       866550
Rfrg_Prvdr_City                           0
Rfrg_Prvdr_State_Abrvtn                   0
Rfrg_Prvdr_State_FIPS                     0
Rfrg_Prvdr_Zip5                           0
Rfrg_Prvdr_RUCA                        1080
Rfrg_Prvdr_RUCA_Desc                   1080
Rfrg_Prvdr_Cntry                          0
Rfrg_Prvdr_Spclty_Desc                    0
Rfrg_Prvdr_Spclty_Srce                    0
Tot_Suplrs                                0
Tot_Suplr_HCPCS_Cds                       0
Tot_Suplr_Benes                           0
Tot_Suplr_Clms                            0
Tot_Suplr_Srvcs                           0
Suplr_Sbmtd_Chrgs               

In [10]:
import re

# replace NaN credentials with empty string
df_clean['Rfrg_Prvdr_Crdntls'] = df_raw['Rfrg_Prvdr_Crdntls'].fillna('')

# normalize credentials to lowercase and remove punctuation
df_clean['Rfrg_Prvdr_Crdntls'] = df_raw['Rfrg_Prvdr_Crdntls'].apply(lambda x: re.sub(r'[^a-zA-Z]','',str(x)).lower())

df_clean.isnull().sum()

Rfrg_NPI                                  0
Rfrg_Prvdr_Last_Name_Org                  0
Rfrg_Prvdr_First_Name                    16
Rfrg_Prvdr_MI                        359299
Rfrg_Prvdr_Crdntls                        0
Rfrg_Prvdr_Ent_Cd                         0
Rfrg_Prvdr_St1                            0
Rfrg_Prvdr_St2                       866550
Rfrg_Prvdr_City                           0
Rfrg_Prvdr_State_Abrvtn                   0
Rfrg_Prvdr_State_FIPS                     0
Rfrg_Prvdr_Zip5                           0
Rfrg_Prvdr_RUCA                        1080
Rfrg_Prvdr_RUCA_Desc                   1080
Rfrg_Prvdr_Cntry                          0
Rfrg_Prvdr_Spclty_Desc                    0
Rfrg_Prvdr_Spclty_Srce                    0
Tot_Suplrs                                0
Tot_Suplr_HCPCS_Cds                       0
Tot_Suplr_Benes                           0
Tot_Suplr_Clms                            0
Tot_Suplr_Srvcs                           0
Suplr_Sbmtd_Chrgs               

### Perform text normalization on provider credentials

In [11]:
# build mapping of zip codes to RUCA
# use 2010 data to keep consistent with methodology per data dictionary
import csv

zip_map = {}

with open('/dsa/groups/casestudycf25/team02/bronze/RUCA2010zipcode.csv', mode='r', newline='') as file:
    csv_reader = csv.DictReader(file)
    # print(csv_reader.fieldnames)
    # row = next(csv_reader)
    # print(re.sub(r"'","",row[csv_reader.fieldnames[0]]))
    for row in csv_reader:
        # write RUCA2 code (value) into zip_map with ZIP_CODE as key
        zip_map[re.sub(r"'","",row[csv_reader.fieldnames[0]])] = float(row['RUCA2']) # Access data using column names

### Use the 2010 Rural-Urban Commuting Area Codes and ZIP codes table from the US Department of Agriculture to build a mapping of zip codes to rural-urban commuting area (RUCA) codes for imputing null RUCA values.

In [12]:
# build mapping of RUCA to urban-rural designation and description per https://www.ers.usda.gov/data-products/rural-urban-commuting-area-codes
code_map = {1.:'Metropolitan area core: primary flow within an urbanized area of 50,000 and greater',
            1.1:'Secondary flow 30% to <50% to a larger urbanized area of 50,000 and greater',
            2.:'Metropolitan area high commuting: primary flow 30% or more to a urbanized area of 50,000 and greater',
            2.1:'Secondary flow 30% to <50% to a larger urbanized area of 50,000 and greater',
            3.:'Metropolitan area low commuting: primary flow 10% to <30% to a urbanized area of 50,000 and greater',
            4.:'Micropolitan area core: primary flow within an urban cluster of 10,000 to 49,999',
            4.1:'Secondary flow 30% to <50% to a urbanized area of 50,000 and greater',
            5.:'Micropolitan high commuting: primary flow 30% or more to a urban cluster of 10,000 to 49,999',
            5.1:'Secondary flow 30% to <50% to a urbanized area of 50,000 and greater',
            6.:'Micropolitan low commuting: primary flow 10% to <30% to a urban cluster of 10,000 to 49,999',
            7.:'Small town core: primary flow within an urban cluster of 2,500 to 9,999',
            7.1:'Secondary flow 30% to <50% to a urbanized area of 50,000 and greater',
            7.2:'Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999',
            8.:'Small town high commuting: primary flow 30% or more to a urban cluster of 2,500 to 9,999',
            8.1:'Secondary flow 30% to <50% to a urbanized area of 50,000 and greater',
            8.2:'Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999',
            9.:'Small town low commuting: primary flow 10% to <30% to a urban cluster of 2,500 to 9,999',
            10.:'Rural areas: primary flow to a tract outside a urbanized area of 50,000 and greater or UC',
            10.1:'Secondary flow 30% to <50% to a urbanized area of 50,000 and greater',
            10.2:'Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999',
            10.3:'Secondary flow 30% to <50% to a urban cluster of 2,500 to 9,999',
            99.:'Unknown'}

In [13]:
missing_zips = df_raw[df_raw.Rfrg_Prvdr_RUCA.isnull()]['Rfrg_Prvdr_Zip5'].unique()

# some zip codes are missing from RUCA2010zipcode.csv
for missing_zip in missing_zips:
    try:
        print(f"{missing_zip} maps to {zip_map[missing_zip]}")
    except:
        zip_map[missing_zip] = 99. # map zip code to 99 for unknown

21264 maps to 1.0
99588 maps to 10.0
38163 maps to 1.0
99574 maps to 10.0
80262 maps to 1.0
23291 maps to 1.0
94161 maps to 1.0
75284 maps to 1.0
22334 maps to 1.0
44193 maps to 1.0
02241 maps to 1.0
32886 maps to 1.0
48277 maps to 1.0
99686 maps to 10.0
32323 maps to 2.0
10087 maps to 1.0
63195 maps to 1.0
06030 maps to 1.0
06927 maps to 1.0
06102 maps to 1.0
06050 maps to 1.0
06409 maps to 1.0
06504 maps to 1.0
06520 maps to 1.0
06856 maps to 1.0
06134 maps to 1.0
06089 maps to 1.0
06250 maps to 4.1
06039 maps to 10.0
06904 maps to 1.0
06777 maps to 10.0
46277 maps to 1.0
06115 maps to 1.0
06487 maps to 1.0
06782 maps to 1.0
44194 maps to 1.0
06230 maps to 2.0
06825 maps to 1.0
06890 maps to 1.0
06824 maps to 1.0
35246 maps to 1.0
19194 maps to 1.0
48267 maps to 1.0


In [14]:
# link missing RUCA by zip code from table at https://www.ers.usda.gov/data-products/rural-urban-commuting-area-codes
        
df_clean.Rfrg_Prvdr_RUCA = df_raw.Rfrg_Prvdr_RUCA.fillna(df_raw.Rfrg_Prvdr_Zip5.map(zip_map))
df_clean.isnull().sum()

Rfrg_NPI                                  0
Rfrg_Prvdr_Last_Name_Org                  0
Rfrg_Prvdr_First_Name                    16
Rfrg_Prvdr_MI                        359299
Rfrg_Prvdr_Crdntls                        0
Rfrg_Prvdr_Ent_Cd                         0
Rfrg_Prvdr_St1                            0
Rfrg_Prvdr_St2                       866550
Rfrg_Prvdr_City                           0
Rfrg_Prvdr_State_Abrvtn                   0
Rfrg_Prvdr_State_FIPS                     0
Rfrg_Prvdr_Zip5                           0
Rfrg_Prvdr_RUCA                           0
Rfrg_Prvdr_RUCA_Desc                   1080
Rfrg_Prvdr_Cntry                          0
Rfrg_Prvdr_Spclty_Desc                    0
Rfrg_Prvdr_Spclty_Srce                    0
Tot_Suplrs                                0
Tot_Suplr_HCPCS_Cds                       0
Tot_Suplr_Benes                           0
Tot_Suplr_Clms                            0
Tot_Suplr_Srvcs                           0
Suplr_Sbmtd_Chrgs               

In [15]:
# map in descriptions
df_clean.Rfrg_Prvdr_RUCA_Desc = df_clean.Rfrg_Prvdr_RUCA_Desc.fillna(df_clean.Rfrg_Prvdr_RUCA.map(code_map))
df_clean.isnull().sum()

Rfrg_NPI                                  0
Rfrg_Prvdr_Last_Name_Org                  0
Rfrg_Prvdr_First_Name                    16
Rfrg_Prvdr_MI                        359299
Rfrg_Prvdr_Crdntls                        0
Rfrg_Prvdr_Ent_Cd                         0
Rfrg_Prvdr_St1                            0
Rfrg_Prvdr_St2                       866550
Rfrg_Prvdr_City                           0
Rfrg_Prvdr_State_Abrvtn                   0
Rfrg_Prvdr_State_FIPS                     0
Rfrg_Prvdr_Zip5                           0
Rfrg_Prvdr_RUCA                           0
Rfrg_Prvdr_RUCA_Desc                      0
Rfrg_Prvdr_Cntry                          0
Rfrg_Prvdr_Spclty_Desc                    0
Rfrg_Prvdr_Spclty_Srce                    0
Tot_Suplrs                                0
Tot_Suplr_HCPCS_Cds                       0
Tot_Suplr_Benes                           0
Tot_Suplr_Clms                            0
Tot_Suplr_Srvcs                           0
Suplr_Sbmtd_Chrgs               

### Append years

In [16]:
import numpy as np
# arrays of year values
yrs_21 = np.full(len21, 2021)
yrs_22 = np.full(len22, 2022)
yrs_23 = np.full(len23, 2023)

# concatenate
yrs = np.concatenate((yrs_21, yrs_22, yrs_23))

# add year column to the df
df_clean['Year'] = yrs

df_clean.head()

,Rfrg_NPI,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,Rfrg_Prvdr_State_Abrvtn,...,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre,Year
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,-1.0,-1.000000,1.000000,-1.0,-1.0,0.0,-1.0,-1.0,1.942167,2021
1,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.278200,2021
2,1003000522,Weigand,Frederick,J,md,I,1565 Saxon Blvd,Suite 102,Deltona,FL,...,-1.0,1.000000,0.846154,-1.0,-1.0,0.0,-1.0,-1.0,1.872308,2021
3,1003000530,Semonche,Amanda,M,do,I,1021 Park Ave,Suite 203,Quakertown,PA,...,-1.0,0.833333,0.777778,-1.0,-1.0,-1.0,-1.0,-1.0,2.144684,2021
4,1003000597,Kim,Dae,Y,mdphd,I,1145 S Utica Ave,Suite 202,Tulsa,OK,...,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.006500,2021


## Write-out and validate combined datasets

In [17]:
# prepare the to_snake_case function
def to_snake_case(name: str) -> str:
    # Add underscore between lower-to-upper transitions
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)

    # Replace non-alphanumeric with underscores
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)

    # Remove leading/trailing underscores and lowercase
    return name.strip("_").lower()

In [18]:
# convert column headings to snake case
df_clean = df_clean.rename(columns={col: to_snake_case(col) for col in df_clean.columns})

In [19]:
# write to csv
df_clean.to_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrr.csv', index=False) 

In [20]:
import pandas as pd

df = pd.read_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrr.csv',dtype={'rfrg_prvdr_state_fips':str,'rfrg_prvdr_zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str
df.head()

,rfrg_npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,rfrg_prvdr_st2,rfrg_prvdr_city,rfrg_prvdr_state_abrvtn,...,bene_cc_ph_hf_non_ihd_v2_pct,bene_cc_ph_hyperlipidemia_v2_pct,bene_cc_ph_hypertension_v2_pct,bene_cc_ph_ischemic_heart_v2_pct,bene_cc_ph_osteoporosis_v2_pct,bene_cc_ph_parkinson_v2_pct,bene_cc_ph_arthritis_v2_pct,bene_cc_ph_stroke_tia_v2_pct,bene_avg_risk_scre,year
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,-1.0,-1.000000,1.000000,-1.0,-1.0,0.0,-1.0,-1.0,1.942167,2021
1,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.278200,2021
2,1003000522,Weigand,Frederick,J,md,I,1565 Saxon Blvd,Suite 102,Deltona,FL,...,-1.0,1.000000,0.846154,-1.0,-1.0,0.0,-1.0,-1.0,1.872308,2021
3,1003000530,Semonche,Amanda,M,do,I,1021 Park Ave,Suite 203,Quakertown,PA,...,-1.0,0.833333,0.777778,-1.0,-1.0,-1.0,-1.0,-1.0,2.144684,2021
4,1003000597,Kim,Dae,Y,mdphd,I,1145 S Utica Ave,Suite 202,Tulsa,OK,...,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.006500,2021


Verify the number of records was preserved.

In [21]:
check_len = len21 + len22 + len23
df_len = len(df)
print(f'{check_len} versus {df_len}')

1157253 versus 1157253
